# 009: BERT Masked Language Modeling — Bidirectional context and MLM training objective

## MLM OBJECTIVE (Devlin et al., 2019)
- **Masked Language Model**: Randomly mask 15% of input tokens. The model predicts the original token.
- Masking strategy: 80% → [MASK], 10% → random token, 10% → unchanged.
- **Bidirectional**: Unlike GPT, BERT sees context from both left and right simultaneously.
- **[CLS] Token**: Prepended to every input; its final hidden state serves as the sentence-level representation.
---


In [ ]:
import numpy as np

class BERTMLMSimulator:
    """Simulates BERT's masked language modeling preprocessing."""
    def __init__(self, vocab, mask_prob=0.15):
        self.vocab = vocab
        self.mask_token = "[MASK]"
        self.cls_token = "[CLS]"
        self.sep_token = "[SEP]"
        self.mask_prob = mask_prob

    def mask_tokens(self, tokens):
        """Apply BERT masking strategy to a token sequence."""
        masked = list(tokens)
        labels = [-1] * len(tokens)  # -1 means not masked
        for i in range(len(tokens)):
            if np.random.random() < self.mask_prob:
                labels[i] = self.vocab.index(tokens[i])
                rand = np.random.random()
                if rand < 0.8:
                    masked[i] = self.mask_token     # 80%: replace with [MASK]
                elif rand < 0.9:
                    masked[i] = np.random.choice(self.vocab)  # 10%: random token
                # else: 10%: keep original
        return masked, labels

    def prepare_input(self, sentence_tokens):
        """Prepare BERT-style input with [CLS] and [SEP]."""
        tokens = [self.cls_token] + sentence_tokens + [self.sep_token]
        masked, labels = self.mask_tokens(tokens)
        return {
            "original": tokens,
            "masked_input": masked,
            "labels": [-1] + labels[1:-1] + [-1]  # Don't mask special tokens
        }



In [ ]:
vocab = ["the", "cat", "sat", "on", "mat", "dog", "ran", "big", "small", "red"]
bert = BERTMLMSimulator(vocab, mask_prob=0.3)

np.random.seed(42)
sentence = ["the", "big", "cat", "sat", "on", "the", "red", "mat"]
result = bert.prepare_input(sentence)

print("--- BERT MLM Preprocessing ---")
print(f"Original: {result['original']}")
print(f"Masked:   {result['masked_input']}")
print(f"Labels:   {result['labels']}")
print("\n[Note] Labels != -1 indicate positions where the model must predict the original token.")



In [ ]:
# ── Multiple masking examples ──
print("\n--- Multiple Masking Variations ---")
for trial in range(3):
    result = bert.prepare_input(sentence)
    masked_positions = [i for i, l in enumerate(result['labels']) if l != -1]
    print(f"Trial {trial+1}: Masked at positions {masked_positions} → {[result['masked_input'][p] for p in masked_positions]}")
